In [ ]:
from google.colab import userdata     # Imports secret tokens for logins
from huggingface_hub import login     # Huggingface for fast implementation of transformers
import os
import sys
from pathlib import Path
file_path = ('/content/drive/My Drive/Thesis/belief-repr-1/')
sys.path.append(file_path)
if not os.path.exists('/content/drive/My Drive'):
    from google.colab import drive
    drive.mount('/content/drive')

secrets = {
    'hugging': userdata.get('hugging_token'),
    'wandb': userdata.get('wandb_api'),
    'nnsight': userdata.get('nnsight'),
    'openai': userdata.get('openai_api')
}

HF_TOKEN = secrets['hugging']
login(token=HF_TOKEN)

%pip install -U datasets einops jaxtyping
%pip install openai

In [ ]:
# import nnsight

# import circuitsvis as cv

''' Tensor manipulation '''

import einops
from einops import einsum
import numpy as np
import torch as t                     # https://pytorch.org/docs/stable/torch.html
import torch.nn as nn                 # https://pytorch.org/docs/stable/nn.html
import torch.nn.functional as F       # https://pytorch.org/docs/stable/nn.functional.html

''' Utils '''

import pandas as pd
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import gc
import seaborn as sns

''' Should we need strong typing '''

from jaxtyping import Float, Int
from torch import Tensor
from typing import Callable, List, Tuple

''' For visualization of progress (in notebook) and training behavior (in wandb) '''

from tqdm import tqdm
import wandb                          # REMEMBER to log training loops if you want to analyze that behavior | how to: wandb.init() before training, wandb.log() at each epoch, wandb.finish() to clean cache
wandb.login(key=secrets['wandb'])
device = t.device("mps" if t.backends.mps.is_available() else "cuda" if t.cuda.is_available() else "cpu")

''' My things '''

from probes import SupervisedProbe
from visualization import *
from hooks import *
from utils import *
from data_sets import *
from intervention import *
from shots import *
from subjprobs import *

In [ ]:
# Inferred from exp 1, best layer for LR probes  
# This is not the best layer for MMP probes, but it's a good enough approximation

best_layer = {'mmp': {
                'llama': 12,
                'llama_instruct': 14,
                'gemma': 25,
                'gemma_instruct': 30,
                'gpt-j': 22
                      },
              'lr': {
                'llama': 15,
                'llama_instruct': 13,
                'gemma': 27,
                'gemma_instruct': 28,
                'gpt-j': 13
                      }
              }

In [ ]:
# linear_layer best lr for residual stream on llama == ~1e-2/1e-3
# mlp best lr for residual stream on llama == ~1e-4
# mmp returns immediate results
# no batching is the fastest and works very well for cities.csv
# diminishing return after the 100th epoch for good activations

class ProbeConfig:
    def __init__(self):
        """ General """
        self.device = device
        self.batch_size_extractor = 128
        self.seed = 42
        """ Probe setup """
        self.probe_type = "mmp" # Options: linear_layer | linear | mlp | mmp
        self.supervision = "S"
        self.direction_type = 'mmp'  # Options: linear | logistic | mmp
        self.verbose = False
        self.with_std = True
        self.var_normalize = True
        self.control = False
        """ Specs """
        self.batch_size = -1
        self.nepochs = 50
        self.ntries = 1
        self.lr = 1e-3
        self.weight_decay = 0.0
        self.dropout = 0.0
        self.C = 1e6
        self.max_iter = 500
        self.test_size = 0.1
        self.log_accuracy_on_recursive = True
        self.patience = 10

probe_config = ProbeConfig()

MODEL = 'llama'

''' Dictionary for all of the models '''

mymodels = {
    'gemma': lambda: tlens.HookedTransformer.from_pretrained("gemma-2-9b", device=t.device('cpu')).half(),
    'gemma_instruct': lambda: tlens.HookedTransformer.from_pretrained("gemma-2-9b-it", device=t.device('cpu')).half(),
    'llama': lambda: tlens.HookedTransformer.from_pretrained("meta-llama/Llama-3.1-8B", device=t.device('cpu')).half(),
    'llama_instruct': lambda: tlens.HookedTransformer.from_pretrained("meta-llama/Llama-3.1-8B-Instruct", device=t.device('cpu')).half(),
    'gpt-j': lambda: tlens.HookedTransformer.from_pretrained("EleutherAI/gpt-j-6B", device=t.device('cpu')).half(),
}

model = mymodels[MODEL]()
model.to(device)

# TO DO: add SAE Hooked models

# Datasets

In [ ]:
CUTOFF = 1500

# Logical - easy
with open(f'{file_path}curated_dataset_full.pkl', 'rb') as file:
    curated_dataset = pickle.load(file)
with open(f'{file_path}neg_dataset.pkl', 'rb') as file:
    negated = pickle.load(file)
with open(f'{file_path}disj_dataset.pkl', 'rb') as file:
    disjunction = pickle.load(file)
with open(f'{file_path}conj_dataset.pkl', 'rb') as file:
    conjunction = pickle.load(file)
# Logical - inference
with open(f'{file_path}larger_than_inference.pkl', 'rb') as file:
    lt_inference = pickle.load(file)
with open(f'{file_path}smaller_than_inference.pkl', 'rb') as file:
    st_inference = pickle.load(file)
with open(f'{file_path}cities_inference.pkl', 'rb') as file:
    cities_inference = pickle.load(file)
with open(f'{file_path}companies_inference.pkl', 'rb') as file:
    companies_inference = pickle.load(file)
with open(f'{file_path}commonclaim_inference.pkl', 'rb') as file:
    cc_inference = pickle.load(file)
with open(f'{file_path}counterfact_inference.pkl', 'rb') as file:
    cf_inference = pickle.load(file)
# Other datasets
with open(f'{file_path}mymulan.pkl', 'rb') as file:
    mulan = pickle.load(file)
with open(f'{file_path}tqa_curated.pkl', 'rb') as file:
    tqa = pickle.load(file)
with open(f'{file_path}likely.pkl', 'rb') as file:
    likely = pickle.load(file)
with open(f'{file_path}entailment_new.pkl', 'rb') as file:
    entailment_new = pickle.load(file)

''' Data cleanup and division '''

# Clean 'negated'

new_column_for_neg = curated_dataset[curated_dataset['filename'].isin(['common_claim_true_false.csv', 'companies_true_false.csv', 'counterfact_true_false.csv'])]
new_column_for_neg['filename'].unique()
negated = negated.rename(columns={'statement':'new_statement'})
negated['statement'] = new_column_for_neg['statement']
negated = negated[['statement', 'new_statement', 'label', 'filename']]
negated['label'] = 1 - negated['label']
negated['neg_label'] = 1 - negated['label']

# Mulan division

mulan_mutable = mulan[mulan['type'] == 'mutable']
mulan_immutable = mulan[mulan['type'] == 'immutable']
mulan_mutable = stratified_sample(mulan_mutable, 'relation', CUTOFF)
mulan_immutable = stratified_sample(mulan_immutable, 'relation', CUTOFF)

In [ ]:
X_hop = None    # Discriminates inference task from others

datasplit = get_data_split('inference', curated_dataset, probe_config=probe_config, other_dataset=entailment_new, cutoff=CUTOFF)

''' 
datasplit is a tuple containing all relevant data for the coherence experiment 
it returns: X_clean_train, y_clean_train, X_clean_test_base, X_clean_test_paraph, y_clean_test_label
in case of inference task (we need another set of statements): X_clean_train, y_clean_train, X_clean_test_base, X_clean_test_paraph, X_clean_test_hop, y_clean_test_label
test labels are helpers for activation extractor, they are not important for our metrics
'''

X_clean_train, y_clean_train, df = datasplit

## Training Loop

# Probe training

In [ ]:
# == Resid

model.reset_hooks()
resid_extractor = ActivationExtractor(model=model, data=X_clean_train, labels=y_clean_train, device=device, half=True,
                                      batch_size=probe_config.batch_size_extractor)
resid_extractor.set_hooks(
                          [best_layer[probe_config.probe_type][MODEL]],
                          [tlens.utils.get_act_name('resid_post')], attn=False) # for instance
resid_activations, resid_labels = resid_extractor.process() # Get

model.to(t.device('cpu'))
gc.collect()
t.cuda.empty_cache()

### LR


In [ ]:
import warnings
warnings.filterwarnings("ignore", message=".*To copy construct from a tensor.*")            # Ignores an annoying warning from sklearn

probe_config.verbose=False
probe_config.control=False
probe_config.lr=0.0009
probe_config.nepochs=1000
probe_config.batch_size=512
probe_config.dropout=0.2

dataset = einops.rearrange(next(iter(resid_activations.values())), 'n b d -> (n b) d')
gold = einops.rearrange(resid_labels, 'n b -> (n b)')
X_train, X_test, y_train, y_test = train_test_split(dataset, gold, test_size=probe_config.test_size, random_state=probe_config.seed)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

master_probe, directions = form_master_probe(probe_config, X_train, y_train, X_test, y_test, n=10)
master_probe.to(device)

### MMP

In [ ]:
# Mass-mean Probe

with t.autocast('cuda'):
  X_train = X_train.astype(np.float32)
  X_test = X_test.astype(np.float32)
  probe_config.var_normalize = False
  mm_probe = SupervisedProbe(x_train=X_train, labels_train=y_train,
                          x_test=X_test, labels_test=y_test,
                          probe_cfg=probe_config)
  mm_probe.repeated_train()
  mm = mm_probe.get_direction()
  mm_acc = mm_probe.get_acc()

In [ ]:
# cleanup resid activations and labels from device if needed

del resid_activations
del resid_labels
gc.collect()
t.cuda.empty_cache()

# Tests - Interp

## Conj/Disj/Neg/CrossDataset

In [ ]:
# Extract representations for test set

X_hop = None

df_a = df[0].iloc[:-(len(df[0]) % probe_config.batch_size_extractor)]
df_b = df[1].iloc[:-(len(df[1]) % probe_config.batch_size_extractor)]

# For cross-dataset task:

'''
X_base = list(df_a['statement'])
X_paraph = list(df_b['statement'])
'''

# for logical non-inference tasks:

X_base = list(df_a['statement'])
X_paraph = list(df_a['new_statement'])
labels = list(df_a['label'])  # Only necessary for neg/conj/disj

# for inference task:
'''
X_base = list(df_a['statement'])
X_paraph = list(df_a['new_statement'])
X_hop = list(df_a['hop_statement'])
'''

In [ ]:
''' Set up X_base and X_paraph '''

model.reset_hooks()
base_extractor = ActivationExtractor(model=model, data=X_base, labels=y_test, device=device, half=True,
                                      batch_size=probe_config.batch_size_extractor)
base_extractor.set_hooks(
                          [best_layer[probe_config.probe_type][MODEL]],
                          [tlens.utils.get_act_name('resid_post')], attn=False) # for instance
base_activations, base_labels = base_extractor.process() # Get
# model.reset_hooks()
paraph_extractor = ActivationExtractor(model=model, data=X_paraph, labels=y_test, device=device, half=True,
                                      batch_size=probe_config.batch_size_extractor)
paraph_extractor.set_hooks(
                          [best_layer[probe_config.probe_type][MODEL]],
                          [tlens.utils.get_act_name('resid_post')], attn=False) # for instance
paraph_activations, paraph_labels = paraph_extractor.process() # Get

base_activations_values = base_activations[next(iter(base_activations))]
base_activations_values = einops.rearrange(base_activations_values, 'n b d -> (n b) d')
base_activations_values = scaler.transform(base_activations_values)
paraph_activations_values = paraph_activations[next(iter(paraph_activations))]
paraph_activations_values = einops.rearrange(paraph_activations_values, 'n b d -> (n b) d')
paraph_activations_values = scaler.transform(paraph_activations_values)

if X_hop is not None:   # Get data for entailment
  model.reset_hooks()
  hop_extractor = ActivationExtractor(model=model, data=X_hop, labels=y_test, device=device, half=True,
                                        batch_size=probe_config.batch_size_extractor)
  hop_extractor.set_hooks(
                            [best_layer[probe_config.probe_type][MODEL]],
                            [tlens.utils.get_act_name('resid_post')], attn=False) # for instance
  hop_activations, hop_labels = hop_extractor.process() # Get

  hop_activations_values = hop_activations[next(iter(hop_activations))]
  hop_activations_values = einops.rearrange(hop_activations_values, 'n b d -> (n b) d')
  hop_activations_values = scaler.transform(hop_activations_values)

model.to(t.device('cpu'))
gc.collect()
t.cuda.empty_cache()

### LR

In [ ]:
master_base_probabilities = master_probe(t.tensor(base_activations_values, device=device, dtype=t.float32))
master_paraph_probabilities = master_probe(t.tensor(paraph_activations_values, device=device, dtype=t.float32))
df['master_label_base'] = master_base_probabilities.cpu().numpy() > 0.5
df['master_label_paraph'] = master_paraph_probabilities.cpu().numpy() > 0.5
df['master_proba_base'] = master_base_probabilities.cpu().numpy()
df['master_proba_paraph'] = master_paraph_probabilities.cpu().numpy()
if X_hop is not None:
  master_hop_probabilities = master_probe(t.tensor(hop_activations_values, device=device, dtype=t.float32))
  df['master_label_hop'] = master_hop_probabilities.cpu().numpy() > 0.5
  df['master_proba_hop'] = master_hop_probabilities.cpu().numpy()

### MMP

In [ ]:
with t.autocast('cuda'):

  mm_base_probabilities = mm_probe.probe(t.tensor(base_activations_values, device=device, dtype=t.float32))
  mm_paraph_probabilities = mm_probe.probe(t.tensor(paraph_activations_values, device=device, dtype=t.float32))
  df['mm_label_base'] = mm_base_probabilities.detach().cpu().numpy() > 0.5
  df['mm_label_paraph'] = mm_paraph_probabilities.detach().cpu().numpy() > 0.5
  df['mm_proba_base'] = mm_base_probabilities.detach().cpu().numpy()
  df['mm_proba_paraph'] = mm_paraph_probabilities.detach().cpu().numpy()
  if X_hop is not None:
    mm_hop_probabilities = mm_probe.probe(t.tensor(hop_activations_values, device=device, dtype=t.float32))
    df['mm_label_hop'] = master_hop_probabilities.detach().cpu().numpy() > 0.5
    df['mm_proba_hop'] = master_hop_probabilities.detach().cpu().numpy()

# Tests - Non-interp

## Self-reporting

In [ ]:
model.reset_hooks()
model.to(device)
def embed_to_half_hook(tensor, hook):
    return tensor.half()

model.add_hook("hook_embed", embed_to_half_hook)

import gc

context = "I am a fact-checking AI. For each statement, I determine whether it is True or False and rate my confidence in that judgment on a scale from 0 to 1."
shots = [
    'Statement: Paris is the capital of France.\nAnswer: True\nConfidence: 0.95',
    'Statement: The largest bear in the world is currently in Italy.\nAnswer: False\nConfidence: 0.75',
    'Statement: Milan is the capital of Italy.\nAnswer: False\nConfidence: 0.85',
    'Statement: Pythagoras was originally from Turkey.\nAnswer: False\nConfidence: 0.70'
]

tqdm.pandas()


In [ ]:
''' Logical '''

df[['self_label_base', 'self_proba_base']] = df['statement'].progress_apply(self_evaluate_row)
df[['self_label_paraph', 'self_proba_paraph']] = df['new_statement'].progress_apply(self_evaluate_row)
if X_hop is not None:
  df[['self_label_hop', 'self_proba_hop']] = df['hop_statement'].progress_apply(self_evaluate_row)

backup_df = df
df = df.dropna()

''' Cross-dataset '''

# df[['self_label_base', 'self_proba_base']] = cut_curated['statement'].apply(self_evaluate_row)
# df[['self_label_base', 'self_proba_base']] = mulan_immutable['statement'].apply(self_evaluate_row)
# df[['self_label_paraph', 'self_proba_paraph']] = cut_tqa['statement'].apply(self_evaluate_row)
# df[['self_label_paraph', 'self_proba_paraph']] = cut_likely['statement'].apply(self_evaluate_row)
# df[['self_label_paraph', 'self_proba_paraph']] = mulan_mutable['statement'].apply(self_evaluate_row)

## Logit-based estimate

In [ ]:
# Apply it to the DataFrame
df[['logit_label_base', 'logit_proba_base']] = df['statement'].progress_apply(logit_evaluate_row)
df[['logit_label_paraph', 'logit_proba_paraph']] = df['new_statement'].progress_apply(logit_evaluate_row) # Fix with the correct column
if X_hop is not None:
  df[['logit_label_hop', 'logit_proba_hop']] = df['hop_statement'].progress_apply(logit_evaluate_row)

''' Cross-dataset '''

# df[['self_label_base', 'self_proba_base']] = cut_curated['statement'].apply(logit_evaluate_row)
# df[['self_label_base', 'self_proba_base']] = mulan_immutable['statement'].apply(logit_evaluate_row)
# df[['self_label_paraph', 'self_proba_paraph']] = cut_tqa['statement'].apply(logit_evaluate_row)
# df[['self_label_paraph', 'self_proba_paraph']] = cut_likely['statement'].apply(logit_evaluate_row)
# df[['self_label_paraph', 'self_proba_paraph']] = mulan_mutable['statement'].apply(logit_evaluate_row)

## We have full probas. Let us turn to evaluation

In [ ]:
random_base = t.rand(len(df))
random_hop = t.rand(len(df))
random_base = np.array(df['logit_proba_base'])
np.random.shuffle(random_base)
random_hop = np.array(df['logit_proba_paraph'])
np.random.shuffle(random_base)

In [ ]:
plot_histogram(np.array(df['self_proba_paraph'].tolist() - np.array(df['self_proba_base'].tolist()) ))

In [ ]:
plot_histogram(((df['logit_proba_base'])), title="Credence distribution (paraph)")

In [ ]:
judge_negation = JudgeCoherence(logic='neg')
judge_disjunction = JudgeCoherence(logic='disj')
judge_conjunction = JudgeCoherence(logic='conj')
judge_ent = JudgeCoherence(logic='ent')
judge_ent_ = JudgeCoherence(logic='ent*')

backup_df = df

# example
# judge_negation.set_metric(judge_negation.rmse_metric)
# performance_negation_master = judge_negation.judge([t.tensor(list(df['master_proba_base'])), t.tensor(list(df['master_proba_paraph']))])
# performance_negation_mm = judge_negation.judge([t.tensor(list(df['mm_proba_base'])), t.tensor(list(df['mm_proba_paraph']))])
# performance_negation_self = judge_negation.judge([t.tensor(list(df['self_proba_base'])), t.tensor(list(df['self_proba_paraph']))])
# performance_negation_logit = judge_negation.judge([t.tensor(list(df['logit_proba_base'])), t.tensor(list(df['logit_proba_paraph']))])
# print(1/(1+performance_negation_master))
# print(1/(1+performance_negation_mm))
# print(1/(1+performance_negation_self))
# print(1/(1+performance_negation_logit))
# judge_disjunction.set_metric(judge_disjunction.less_than_perc)
# performance_disjunction_master = judge_disjunction.judge([t.tensor(list(df['master_proba_base'])), t.tensor(list(df['master_proba_paraph']))])
# performance_disjunction_mm = judge_disjunction.judge([t.tensor(list(df['mm_proba_base'])), t.tensor(list(df['mm_proba_paraph']))])
# performance_disjunction_self = judge_disjunction.judge([t.tensor(list(df['self_proba_base'])), t.tensor(list(df['self_proba_paraph']))])
# performance_disjunction_logit = judge_disjunction.judge([t.tensor(list(df['logit_proba_base'])), t.tensor(list(df['logit_proba_paraph']))])
# print(performance_disjunction_master)
# print(performance_disjunction_mm)
# print(performance_disjunction_self)
# print(performance_disjunction_logit)
judge_disjunction.set_metric(judge_disjunction.less_than_perc)
performance_disjunction_master = judge_disjunction.judge([t.tensor(list(df['master_proba_base'])), t.tensor(list(df['master_proba_paraph']))])
performance_disjunction_mm = judge_disjunction.judge([t.tensor(list(df['mm_proba_base'])), t.tensor(list(df['mm_proba_paraph']))])
performance_disjunction_self = judge_disjunction.judge([t.tensor(list(df['self_proba_base'])), t.tensor(list(df['self_proba_paraph']))])
performance_disjunction_logit = judge_disjunction.judge([t.tensor(list(df['logit_proba_base'])), t.tensor(list(df['logit_proba_paraph']))])
print(1-performance_disjunction_master)
print(1-performance_disjunction_mm)
print(1-performance_disjunction_self)
print(1-performance_disjunction_logit)
# judge_conjunction.set_metric(judge_conjunction.less_than_perc)
# performance_conjunction = judge_conjunction.judge([t.tensor(list(df['mm_proba_paraph'])), t.tensor(list(df['mm_proba_base']))])
# judge_cross_dataset.set_metric(judge_cross_dataset.avg_conf_diff)
# performance_cross_dataset = judge_cross_dataset.judge(df['logit_proba_base'], df['logit_proba_paraph'])
# judge_ent.set_metric(judge_ent.less_than_perc)
# performance_ent = judge_ent.judge([t.tensor(list(df['self_proba_base'])), t.tensor(list(df['self_proba_hop'])), t.tensor(list(df['self_proba_paraph']))])
# judge_ent_.set_metric(judge_ent.less_than_perc)
# performance_ent_ = judge_ent_.judge([t.tensor(list(df['self_proba_base'])), t.tensor(list(df['self_proba_paraph']))])

In [ ]:
df['mm_proba_base'] = df['mm_proba_base'].clip(lower=0.01)
df['mm_proba_paraph'] = df['mm_proba_paraph'].clip(lower=0.01)
df['mm_proba_hop'] = df['mm_proba_hop'].clip(lower=0.01)
df['master_proba_base'] = df['master_proba_base'].clip(lower=0.01)
df['master_proba_paraph'] = df['master_proba_paraph'].clip(lower=0.01)
df['master_proba_hop'] = df['master_proba_hop'].clip(lower=0.01)
df['logit_proba_base'] = df['logit_proba_base'].clip(lower=0.01)
df['logit_proba_paraph'] = df['logit_proba_paraph'].clip(lower=0.01)
df['logit_proba_hop'] = df['logit_proba_hop'].clip(lower=0.01)

In [ ]:
print('performances lr', judge_ent.mae_metric_clamp(df['lr_proba_paraph'], df['lr_proba_base'], df['lr_proba_hop'] 'master'))
print('performances mm', get_perfs(df, 'mm'))
print('performances self', get_perfs(df, 'self'))
print('performances logit', get_perfs(df, 'logit'))

In [ ]:
df_a['self_proba_base'] = df_a['self_proba_base'].apply(extract_confidence)
df_b['self_proba_paraph'] = df_b['self_proba_paraph'].apply(extract_confidence)
df_a = df_a[df_a['self_proba_base'] < 1]
df_a = df_a.dropna()
df_b = df_b[df_b['self_proba_paraph'] < 1]
df_b = df_b.dropna()
# df_a['master_proba_base'] = df_a['master_proba_base'].apply(extract_confidence)
# df_b['master_proba_paraph'] = df_b['master_proba_paraph'].apply(extract_confidence)
# df_a['mm_proba_base'] = df_a['mm_proba_base'].apply(extract_confidence)
# df_b['mm_proba_paraph'] = df_b['mm_proba_paraph'].apply(extract_confidence)
# df_a['logit_proba_base'] = df_a['logit_proba_base'].apply(extract_confidence)
# df_b['logit_proba_paraph'] = df_b['logit_proba_paraph'].apply(extract_confidence)

performance_cross_dataset_self = (t.tensor(list(df_a['self_proba_base'])).mean() - t.tensor(list(df_b['self_proba_paraph'])).mean()).item()
# performance_cross_dataset_master = (t.tensor(list(df_a['master_proba_base'])) - t.tensor(list(df_b['master_proba_paraph']))).mean().item()
# performance_cross_dataset_mm = (t.tensor(list(df_a['mm_proba_base'])) - t.tensor(list(df_b['mm_proba_paraph']))).mean().item()
# performance_cross_dataset_logit = (t.tensor(list(df_a['logit_proba_base'])) - t.tensor(list(df_b['logit_proba_paraph']))).mean().item()

# Calibration study

In [ ]:
df_backup = df
df = df.dropna()

In [ ]:
def pcorrect(proba):

  return max(proba, 1-proba)

def preddy(stringz):
    if stringz == 'True':
        return 1
    elif stringz == 'False':
        return 0
    else:
        return 0

y_pred = df['logit_label_base'].apply(preddy)
y_true = df['label']
conf_master = df['master_proba_base']
conf_mm = df['mm_proba_base'].astype(float)
conf_logit = df['logit_proba_base']
conf_self = df['self_proba_base']
corrects = (1 == y_pred).astype(int)
cmapz = {
    'master': 'Blues',
    'mm': 'Greens',
    'logits': 'YlOrBr',
    'self': 'Purples'
}

In [ ]:
def calibration_curve(confidences, corrects, n_bins=10, quantile_binning=True):
    # Compute adjusted accuracy per prediction:
    # If the prediction was correct, use its confidence.
    # If incorrect, use 1 - confidence (reflects how wrong it was).
    # adjusted_acc = corrects.astype(float)
    # adjusted_acc = np.where(corrects == 1, confidences, 1-confidences)

    # Create a DataFrame with confidence and adjusted accuracy
    df = pd.DataFrame({'conf': confidences, 'adjusted_acc': corrects.astype(int)})

    # Bin the data: either using quantile binning or uniform binning
    if quantile_binning:
        # Quantile binning: same number of samples per bin (as much as possible)
        # pd.qcut returns bin labels and bin edges
        bins = pd.qcut(df['conf'], q=n_bins, retbins=True, duplicates='drop')
        df['bin'] = bins[0].cat.codes       # Assign numeric bin labels
        bin_edges = bins[1]                 # Get bin edge values
    else:
        # Uniform binning: divide [0,1] into equal-width intervals
        bin_edges = np.linspace(0, 1, n_bins + 1)
        df['bin'] = pd.cut(df['conf'], bins=bin_edges, labels=False, include_lowest=True)

    # Group by bin and compute statistics:
    # - average confidence in bin
    # - average adjusted accuracy in bin
    # - count of samples in bin
    bin_stats = df.groupby('bin').agg(
        avg_conf=('conf', 'mean'),
        accuracy=('adjusted_acc', 'mean'),
        count=('conf', 'count')
    ).reset_index()

    # Return the bin statistics and bin edges
    return bin_stats, bin_edges

def compute_ece(bin_stats):
    total = bin_stats['count'].sum()
    ece = ((bin_stats['count'] / total) * np.abs(bin_stats['avg_conf'] - bin_stats['accuracy'])).sum()
    return ece

def compute_mce(y_true, y_prob, bins):
    bin_indices = np.digitize(y_prob, bins) - 1
    mce = 0
    for i in range(len(bins) - 1):
        in_bin = bin_indices == i
        if np.any(in_bin):
            acc = np.mean(y_true[in_bin])
            conf = np.mean(y_prob[in_bin])
            gap = abs(acc - conf)
            mce = max(mce, gap)
    return mce

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors

def compute_ece_mce(bin_stats):
    bin_weights = bin_stats['count'] / bin_stats['count'].sum()
    gaps = (bin_stats['accuracy'] - bin_stats['avg_conf']).abs()

    ece = (bin_weights * gaps).sum()
    mce = gaps.max()

    return ece, mce

def plot_reliability_diagram(bin_stats, bin_edges, ax, title, cmap='Blues'):
    # Calculate bin widths and midpoints
    bin_widths = np.diff(bin_edges)
    bin_centers = bin_edges[:-1] + bin_widths / 2

    # Compute bin error magnitude
    errors = np.abs(bin_stats['accuracy'] - bin_stats['avg_conf'])

    # Normalize errors for colormap scaling
    norm = mcolors.Normalize(vmin=0, vmax=0.3)
    color_map = cm.get_cmap(cmap)

    # Plot perfect calibration line
    ax.plot([0, 1], [0, 1], linestyle='--', color='gray', linewidth=1)

    # Bar colors based on calibration error
    bar_colors = color_map(norm(errors.values))

    # Plot dynamic-width bars
    ax.bar(
        bin_centers,
        bin_stats['accuracy'],
        width=bin_widths,
        align='center',
        alpha=0.9,
        edgecolor='black',
        color=bar_colors,
        linewidth=1
    )

    # Axis and title
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_xlabel('Subjective Probability P(True)', fontsize=12)
    ax.set_ylabel('% True Answers', fontsize=12)
    ax.set_title(title, fontsize=14)

    # Grid
    ax.grid(True, linestyle=':', linewidth=0.6, alpha=0.6)

    # ECE and MCE annotations
    ece, mce = compute_ece_mce(bin_stats)

    ax.text(
        0.05, 0.9,
        f"ECE = {ece:.3f}\nMCE = {mce:.3f}",
        fontsize=11,
        bbox=dict(facecolor='white', edgecolor='gray', boxstyle='round,pad=0.3', alpha=0.8)
    )

    # Optional: add colorbar legend for calibration error
    sm = cm.ScalarMappable(cmap=color_map, norm=norm)
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=ax, pad=0.02)
    cbar.set_label('Calibration Error', fontsize=10)


In [ ]:
y_pred = (conf_logit > 0.55).astype(int)

In [ ]:
np.mean(conf_logit)

In [ ]:
y_pred

In [ ]:
bin_stats, bin_edges = calibration_curve(conf_self, y_pred, n_bins=10, quantile_binning=True)

fig, ax = plt.subplots(figsize=(8, 6))

# Plot the reliability diagram
plot_reliability_diagram(
    bin_stats=bin_stats,
    bin_edges=bin_edges,
    ax=ax,
    title="Calibration (Self)",
    cmap=cmapz['self']  # or any other colormap like 'coolwarm', 'Blues', etc.
)

plt.tight_layout()
plt.show()

# Analysis

### Legacy tests

In [ ]:
cossims = []

for direction in directions:

    cossims.append(cosine_similarity(direction.reshape(1, -1), mm.reshape(1, -1))[0])

print(np.array(cossims).mean())

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def plot_cosine_similarity_heatmap(sim_matrix, labels=None):
    plt.figure(figsize=(10, 8))

    ax = sns.heatmap(
        sim_matrix,
        xticklabels=labels,
        yticklabels=labels,
        cmap="mako",
        annot=True,
        fmt=".3f",
        vmin=0.9,
        vmax=1.000,
        linewidths=0.5,
        linecolor='white',
        square=True,
        cbar_kws={"shrink": 1.00}
    )

    ax.set_title("Pairwise Cosine Similarity LRs", fontsize=16, weight='bold', pad=20)
    ax.set_xlabel("Direction", fontsize=12)
    ax.set_ylabel("Direction", fontsize=12)

    # Rotate x-axis labels for better readability
    plt.xticks(rotation=45, ha='right', fontsize=10)
    plt.yticks(fontsize=10)

    plt.tight_layout()
    plt.show()
def cosine_matrix(vectors):

    similarity_matrix = cosine_similarity(vectors)
    labels = [f'D{i}' for i in range(len(vectors))]
    plot_cosine_similarity_heatmap(similarity_matrix, labels)
    return similarity_matrix

similarity_matrix = cosine_matrix(directions)

In [ ]:
accuracies_simple = orthogonal_probing(probe_config, X_train, y_train, X_test, y_test, n=20, fix_direction=None)

In [ ]:
accuracies_mm = orthogonal_probing(probe_config, X_train, y_train, X_test, y_test, n=19, fix_direction=mm)

accuracies_mm.insert(0, mm_acc)

In [ ]:
master_probe_direction = next(param for param in master_probe.parameters()).cpu().numpy()
accuracies_master = orthogonal_probing(probe_config, X_train, y_train, X_test, y_test, n=19, fix_direction=master_probe_direction)
accuracies_master.insert(0, accuracy) # accuracy = accuracy for the master probe

In [ ]:
accuracies_master[0] = accuracies_master[0].cpu().numpy()

In [ ]:
accuracies = {'Mass-mean': accuracies_mm, 'LR': accuracies_simple, 'Average LR': accuracies_master}

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Optional: Set seaborn style for better aesthetics
sns.set(style="whitegrid", context="talk", palette="deep")

plt.figure(figsize=(8, 6))  # Wider figure for better spacing

for key, value in accuracies.items():
    plt.plot(value, label=key, linewidth=2, marker='o', markersize=7, alpha=0.9)

plt.xlabel('Probing Iteration', fontsize=14)
plt.ylabel('Probe Accuracy', fontsize=14)
plt.title('Orthogonal Probing', fontsize=16, weight='bold')
plt.axhspan(0.45, 0.55, color='grey', alpha=0.15, zorder=0)
plt.xticks(fontsize=12, ticks=range(20))
plt.yticks(fontsize=12)
plt.legend(title="Probe", fontsize=11, title_fontsize=12, loc='best')
plt.grid(True, which='both', linestyle='--', linewidth=0.5, alpha=0.8)

plt.tight_layout()
plt.show()


In [ ]:
ex_prompt = 'If you eat a grape you die. This statement is:'
with t.no_grad():
  truth, conf = self_reporting_confidence(model, ex_prompt, context='Use the whole range from 0 to 1 to express precise confidence.')
  print(truth, conf)

# Adapt to whole dataframe

In [ ]:

import torch as t

randt0 = t.rand(1950)
randt1 = t.rand(1950)

(proba1 - proba2).abs().mean().item()